In [ ]:
# ---
# Extract Variants from Annotated VCF Based on Gene Priority
# Priority: Exonic > Intronic > Upstream/Downstream > Intergenic
# Retain only real variants (GT > 0)
# Handle malformed records, redundant variants, and format gene names
# ---


In [ ]:

import pysam
import re
import pandas as pd


# File Paths (Set Yours)
vcf_file = "data/haploid_vcf_ann.vcf"
output_raw = "analysis_results/variants_analysis/files/extracted_variants.tsv"
output_clean = "analysis_results/variants_analysis/files/cleaned_extracted_variants.tsv"

# Read VCF & Initialize
vcf = pysam.VariantFile(vcf_file)
seen_variants = set()  # To avoid duplicate entries
filtered_data = []

# Process Variants by Priority
for record in vcf:
    chrom, pos, ref = record.chrom, record.pos, record.ref
    var_type = record.info.get("TYPE", ["Unknown"])[0]
    ann_info = record.info.get("ANN")

    # Default gene assignment
    closest_gene = "Unknown"
    closest_type = "Unknown"
    closest_distance = float("inf")
    intergenic_gene = None

    if ann_info:
        for annotation in ann_info:
            fields = annotation.split("|")
            if len(fields) < 15:
                print(f"⚠️ Skipping malformed annotation: {annotation}")
                continue

            effect_type = fields[1]
            gene_name = fields[3] or "Unknown"
            distance_info = fields[14].strip()

            # Safe distance conversion
            try:
                distance = abs(int(distance_info)) if distance_info.isdigit() else float("inf")
            except ValueError:
                distance = float("inf")

            # 🧬 Prioritize coding regions
            if effect_type in [
                "disruptive_inframe_insertion", "disruptive_inframe_deletion",
                "frameshift_variant", "start_lost", "stop_gained", "stop_lost",
                "missense_variant", "synonymous_variant", "non_coding_transcript_variant"
            ]:
                closest_gene, closest_type, closest_distance = gene_name, effect_type, 0
                break  # Use first hit in coding regions

            # Next priority: upstream/downstream if closer
            elif effect_type in ["upstream_gene_variant", "downstream_gene_variant"]:
                if distance < closest_distance:
                    closest_gene, closest_type, closest_distance = gene_name, effect_type, distance

            # Lowest priority: intergenic region
            elif effect_type == "intergenic_region":
                intergenic_gene = gene_name

    # Fallback if no gene was selected
    if closest_gene == "Unknown" and intergenic_gene:
        closest_gene = intergenic_gene
        closest_type = "intergenic_region"
        closest_distance = float("inf")

    # Process valid genotypes
    for sample in record.samples:
        genotype = record.samples[sample]["GT"]
        if any(gt > 0 for gt in genotype if gt is not None):  # Only keep variant-present samples
            for alt in record.alts:
                variant_key = (chrom, pos, ref, alt, sample)
                if variant_key not in seen_variants:
                    seen_variants.add(variant_key)
                    filtered_data.append([
                        chrom, pos, ref, alt, var_type, closest_gene,
                        closest_type, closest_distance, sample
                    ])

# Save Initial Results
df = pd.DataFrame(filtered_data, columns=[
    "Chromosome", "Position", "Ref", "Alt", "Variant_Type",
    "Gene", "Annotation_Type", "Distance", "Sample"
])

df.to_csv(output_raw, sep="\t", index=False)
print(f"✅ Extracted variants saved to {output_raw}")

# Clean/Format Gene Names
df["Gene"] = df["Gene"].str.replace(r"^exon-", "", regex=True)
df["Gene"] = df["Gene"].str.replace(r"\.\d+-\d+$", "", regex=True)

# Handle intergenic gene pairs: choose most informative name
def extract_primary_gene(gene_name):
    if pd.isna(gene_name):
        return "Unknown"
    if "-" in gene_name:
        gene_options = gene_name.split("-")
        pmug = next((g for g in gene_options if g.startswith("PmUG01_")), None)
        xm = next((g for g in gene_options if g.startswith("XM_")), None)
        return pmug or (re.sub(r"\.\d+$", "", xm) if xm else gene_name)
    return gene_name

df["Gene"] = df["Gene"].apply(extract_primary_gene)

# Save Cleaned Results
df.to_csv(output_clean, sep="\t", index=False)
print(f"✅ Cleaned extracted variants saved to {output_clean}")
